In [1]:
!pip install flask-ngrok
!pip install pyngrok
!pip install torch transformers
!pip install transformers
!pip install javalang
import nltk
import pandas as pd
import torch
import javalang
import numpy as np
import joblib
import ast
from sklearn.preprocessing import StandardScaler
from javalang.ast import Node
from javalang.parser import Parser
from javalang.tokenizer import tokenize
from flask import Flask, render_template, request
from flask_ngrok import run_with_ngrok
from werkzeug.utils import secure_filename
from pyngrok import ngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 108.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 91.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 93.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitli

In [2]:
!pip install Flask pyngrok
!ngrok authtoken 2jG6PIOLKOFsTOVbCGR9hPCmUCX_4V9xU37QAKjiXTkkUqsqH
from flask import Flask
from pyngrok import ngrok
!pip install flask-ngrok
!pip install pyngrok
!pip install torch transformers
!pip install transformers
!pip install javalang
import javalang
import torch
from transformers import AutoTokenizer, AutoModel
import pandas as pd
import numpy as np
import pickle
import joblib
import ast
from sklearn.preprocessing import StandardScaler
from javalang.ast import Node
from javalang.parser import Parser
from flask import Flask, render_template, request
from flask_ngrok import run_with_ngrok
from werkzeug.utils import secure_filename

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [3]:

#make folder templates and put index.html only.
#upload index.html file from template folder on the labtop
!mkdir templates

In [ ]:
#test classes methods split

In [4]:
def get_class_body(class_node, codelines):
    class_start = class_node.position.line - 1
    brace_count = 0

    for index in range(class_start, len(codelines)):
        line = codelines[index]
        brace_count += line.count('{')
        brace_count -= line.count('}')
        if brace_count == 0:
            class_end = index + 1
            break

    class_body = '\n'.join(codelines[class_start:class_end])
    return class_body.strip()



def process_file(file_path):
    with open(file_path, 'r') as r:
      codelines = r.readlines()
      code_text = ''.join(codelines)

    #tokens = javalang.tokenizer.tokenize(code_text)
    #parser = javalang.parser.Parser(tokens)
    #tree = parser.parse_member_declaration()
    tree = javalang.parse.parse(code_text)
    classes = []

    for _, class_node in tree.filter(javalang.tree.ClassDeclaration):
      class_name = class_node.name
      class_body = get_class_body(class_node, codelines)
      classes.append({'class_name': class_name, 'class_code_snippet': class_body})

    classes_new_data = pd.DataFrame(classes)
    return (classes_new_data)


In [5]:
print(process_file("TestJavaFile4.java"))

             class_name                                 class_code_snippet
0  MailChimpServiceImpl  public class MailChimpServiceImpl implements M...


In [6]:

def get_method_start_end(method_node, tree):
    startpos = None
    endpos = None
    startline = None
    endline = None
    for path, node in tree:
        if startpos is not None and method_node not in path:
            endpos = node.position
            endline = node.position.line if node.position is not None else None
            break
        if startpos is None and node == method_node:
            startpos = node.position
            startline = node.position.line if node.position is not None else None
    return startpos, endpos, startline, endline

def get_method_text(startpos, endpos, startline, endline, last_endline_index, codelines):
    if startpos is None:
        return "", None, None, None
    else:
        startline_index = startline - 1
        endline_index = endline - 1 if endpos is not None else None

        # 1. check for and fetch annotations
        if last_endline_index is not None:
            for line in codelines[(last_endline_index + 1):(startline_index)]:
                if "@" in line:
                    startline_index = startline_index - 1
        meth_text = "<ST>".join(codelines[startline_index:endline_index])
        meth_text = meth_text[:meth_text.rfind("}") + 1]

        # 2. remove trailing rbrace for last methods & any external content/comments
        # if endpos is None and
        if not abs(meth_text.count("}") - meth_text.count("{")) == 0:
            # imbalanced braces
            brace_diff = abs(meth_text.count("}") - meth_text.count("{"))

            for _ in range(brace_diff):
                meth_text = meth_text[:meth_text.rfind("}")]
                meth_text = meth_text[:meth_text.rfind("}") + 1]

        meth_lines = meth_text.split("<ST>")
        meth_text = "".join(meth_lines)
        last_endline_index = startline_index + (len(meth_lines) - 1)

        return meth_text, (startline_index + 1), (last_endline_index + 1), last_endline_index


In [7]:
def process_file(file_path):
    with open(file_path, 'r') as r:
        codelines = r.readlines()
        code_text = ''.join(codelines)

    tree = javalang.parse.parse(code_text)
    methods = {}
    lex = None
    for _, method_node in tree.filter(javalang.tree.MethodDeclaration):
        startpos, endpos, startline, endline = get_method_start_end(method_node, tree)
        method_text, startline, endline, lex = get_method_text(startpos, endpos, startline, endline, lex, codelines)
        methods[method_node.name] = method_text

    # Create a new DataFrame to store the extracted methods
    methods_new_data = pd.DataFrame(columns=['method_name', 'method_code_snippet'])
    methods_new_data['method_name'] = list(methods.keys())
    methods_new_data['method_code_snippet'] = list(methods.values())

    return methods_new_data

In [8]:
print(process_file("TestJavaFile4.java"))

                                method_name  \
0                               getAllLists   
1                               addToMCList   
2                          removeFromMCList   
3                     unsubscribeFromMCList   
4                 updateMCProfileProperties   
5   addProfilePropertiesToMergeFieldsObject   
6                                formatDate   
7                       getMCListProperties   
8                            initHttpClient   
9            isMailChimpConnectorConfigured   
10                  isMemberOfMailChimpList   
11                       updateSubscription   
12                               getBaseUrl   
13                               getHeaders   
14                                setApiKey   
15                          setUrlSubDomain   
16                 setListMergeFieldMapping   
17                 setIsMergeFieldsActivate   

                                  method_code_snippet  
0       public List<HashMap<String, String>> getAl

In [9]:
# web app using flask and ngrok to run the app from google virsual machine
# i think ngrok service help to run your app in your local machine from any where (provide security to this process)
# i think flask provide api to carry the request and the response (carry the snippets to the server or local server in my case and when model pkl
# detect the result then the api carry the response to the machine asked)


In [ ]:
# try use all models sucess on class level only not method level

app = Flask(__name__, template_folder='/content/templates')
# ngrok.set_auth_token('2RQiWOdFHplAaSpQUCH9WOV1auv_2ceqcvPyqL8Wz7uQPBWVN')
# run_with_ngrok(app)

#preprocess class snippets
#count_methods
def count_methods(code):
    # Tokenize and parse the evaluated code
    tokens = javalang.tokenizer.tokenize(code)
    parser = javalang.parser.Parser(tokens)
    tree = parser.parse_member_declaration()
    method_count = 0
    for path, node in tree:
        if isinstance(node, javalang.tree.MethodDeclaration):
            method_count += 1
    return method_count

#count_fields
def count_fields(code):
    # Tokenize and parse the evaluated code
    tokens = javalang.tokenizer.tokenize(code)
    parser = javalang.parser.Parser(tokens)
    tree = parser.parse_member_declaration()
    field_count = 0
    for path, node in tree:
        if isinstance(node, javalang.tree.FieldDeclaration):
            field_count += len(node.declarators)
    return field_count

def calculate_tcc(code):
    # Tokenize and parse the evaluated code
    tokens = javalang.tokenizer.tokenize(code)
    parser = javalang.parser.Parser(tokens)
    tree = parser.parse_member_declaration()

    method_pairs = 0
    common_method_pairs = 0

    for path, node in tree:
        if isinstance(node, javalang.tree.MethodDeclaration):
            method_pairs += 1
            method_invocations = set()
            for _, invocation_node in node.filter(javalang.tree.MethodInvocation):
                method_invocations.add(invocation_node.member)

            for _, other_node in tree:
                if isinstance(other_node, javalang.tree.MethodDeclaration):
                    if other_node.name != node.name:
                        other_method_invocations = set()
                        for _, invocation_node in other_node.filter(javalang.tree.MethodInvocation):
                            other_method_invocations.add(invocation_node.member)
                        if method_invocations.intersection(other_method_invocations):
                            common_method_pairs += 1

    if method_pairs > 0:
        tcc = common_method_pairs / method_pairs
    else:
        tcc = 0

    return tcc


def count_lines_of_code(code):
    # Split the code into lines
    lines = code.split('\n')

    # Count the non-empty lines of code
    loc = sum(1 for line in lines if line.strip() != '')

    return loc

def calculate_atfd(code):
    # Tokenize and parse the evaluated code
    tokens = javalang.tokenizer.tokenize(code)
    parser = javalang.parser.Parser(tokens)
    tree = parser.parse_member_declaration()

    atfd = 0  # Initialize the ATFD metric to 0

    field_declarations = set()  # Store the field declarations in the class

    for path, node in tree:
        if isinstance(node, javalang.tree.FieldDeclaration):
            # Add the field names to the set of field declarations
            for declarator in node.declarators:
                field_declarations.add(declarator.name)

        if isinstance(node, javalang.tree.MethodDeclaration):
            for parameter in node.parameters:
                # Check if the parameter type is from an unrelated class
                if parameter.type.name not in field_declarations:
                    atfd += 1  # Increment the ATFD metric

            if node.body is not None:
                for statement in node.body:
                    # Check if the statement is a method invocation
                    if isinstance(statement, javalang.tree.MethodInvocation):
                        # Check if the invoked method belongs to an unrelated class
                        if statement.member not in field_declarations:
                            atfd += 1  # Increment the ATFD metric

    return atfd



def calculate_cyclomatic_complexity(code):
    try:
        # Tokenize and parse the evaluated code
        tokens = javalang.tokenizer.tokenize(code)
        parser = javalang.parser.Parser(tokens)
        tree = parser.parse_member_declaration()

        # Count the number of decision points (branches and loops)
        complexity = 1  # Start with a complexity of 1 for the method itself
        for _, node in tree:
            if isinstance(node, javalang.tree.IfStatement):
                complexity += 1
            elif isinstance(node, javalang.tree.ForStatement):
                complexity += 1
            elif isinstance(node, javalang.tree.WhileStatement):
                complexity += 1
            elif isinstance(node, javalang.tree.DoStatement):
                complexity += 1
            elif isinstance(node, javalang.tree.SwitchStatement):
                complexity += len(node.cases)

        return complexity

    except (SyntaxError, ValueError) as e:
        print("Error evaluating code:")
        print(code)
        raise e


def calculate_lcom5(code):
    # Tokenize and parse the code snippet
    tokens = javalang.tokenizer.tokenize(code)
    parser = javalang.parser.Parser(tokens)
    tree = parser.parse_member_declaration()

    # Collect all the method names and instance variables
    methods = {}
    variables = set()
    for _, node in tree:
        if isinstance(node, javalang.tree.MethodDeclaration):
            methods[node.name] = set()
            for path, child_node in node:
                if isinstance(child_node, javalang.tree.VariableDeclarator):
                    variables.add(child_node.name)
                    methods[node.name].add(child_node.name)

    # Calculate the LCOM5 metric
    lcom5 = 0
    for method1, vars1 in methods.items():
        for method2, vars2 in methods.items():
            if method1 != method2:
                common_vars = vars1.intersection(vars2)
                if len(common_vars) == 0:
                    lcom5 += 1

    return lcom5




#preprocess methods snippets

# Load the model and tokenizer
model_name = "claudios/cubert-20210711-Java-1024"
model = AutoModel.from_pretrained(model_name)
java_tokenizer = AutoTokenizer.from_pretrained("CAUKiel/JavaBERT")

# Ensure the model is in evaluation mode
model.eval()

# Check if GPU is available and move the model to GPU if possible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Function to get embeddings for a batch of code snippets
def get_code_embeddings_batch(code_snippets):
    inputs = java_tokenizer(code_snippets, return_tensors="pt", truncation=True, padding='max_length', max_length=1024)
    inputs = {key: value.to(device) for key, value in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()  # Use mean pooling for a fixed-size representation
    return embeddings



def get_method_start_end(method_node, tree):
    startpos = None
    endpos = None
    startline = None
    endline = None
    for path, node in tree:
        if startpos is not None and method_node not in path:
            endpos = node.position
            endline = node.position.line if node.position is not None else None
            break
        if startpos is None and node == method_node:
            startpos = node.position
            startline = node.position.line if node.position is not None else None
    return startpos, endpos, startline, endline

def get_method_text(startpos, endpos, startline, endline, last_endline_index, codelines):
    if startpos is None:
        return "", None, None, None
    else:
        startline_index = startline - 1
        endline_index = endline - 1 if endpos is not None else None

        # 1. check for and fetch annotations
        if last_endline_index is not None:
            for line in codelines[(last_endline_index + 1):(startline_index)]:
                if "@" in line:
                    startline_index = startline_index - 1
        meth_text = "<ST>".join(codelines[startline_index:endline_index])
        meth_text = meth_text[:meth_text.rfind("}") + 1]

        # 2. remove trailing rbrace for last methods & any external content/comments
        # if endpos is None and
        if not abs(meth_text.count("}") - meth_text.count("{")) == 0:
            # imbalanced braces
            brace_diff = abs(meth_text.count("}") - meth_text.count("{"))

            for _ in range(brace_diff):
                meth_text = meth_text[:meth_text.rfind("}")]
                meth_text = meth_text[:meth_text.rfind("}") + 1]

        meth_lines = meth_text.split("<ST>")
        meth_text = "".join(meth_lines)
        last_endline_index = startline_index + (len(meth_lines) - 1)

        return meth_text, (startline_index + 1), (last_endline_index + 1), last_endline_index

def process_file(file_path):
    with open(file_path, 'r') as r:
        codelines = r.readlines()
        code_text = ''.join(codelines)

    tree = javalang.parse.parse(code_text)
    methods = {}
    lex = None
    for _, method_node in tree.filter(javalang.tree.MethodDeclaration):
        startpos, endpos, startline, endline = get_method_start_end(method_node, tree)
        method_text, startline, endline, lex = get_method_text(startpos, endpos, startline, endline, lex, codelines)
        methods[method_node.name] = method_text

    # Create a new DataFrame to store the extracted methods
    methods_new_data = pd.DataFrame(columns=['method_name', 'method_code_snippet'])
    methods_new_data['method_name'] = list(methods.keys())
    methods_new_data['method_code_snippet'] = list(methods.values())

    return methods_new_data

def get_class_body(class_node, codelines):
    class_start = class_node.position.line - 1
    brace_count = 0

    for index in range(class_start, len(codelines)):
        line = codelines[index]
        brace_count += line.count('{')
        brace_count -= line.count('}')
        if brace_count == 0:
            class_end = index + 1
            break

    class_body = '\n'.join(codelines[class_start:class_end])
    return class_body.strip()

def god_process_file(file_path):
    with open(file_path, 'r') as r:
      codelines = r.readlines()
      code_text = ''.join(codelines)

    tree = javalang.parse.parse(code_text)
    classes = []

    for _, class_node in tree.filter(javalang.tree.ClassDeclaration):
      class_name = class_node.name
      class_body = get_class_body(class_node, codelines)
      classes.append({'class_name': class_name, 'class_code_snippet': class_body})

    classes_new_data = pd.DataFrame(classes)
    return classes_new_data



@app.route('/')
def index():
    return render_template('index.html')

@app.route('/upload', methods=['POST'])
def upload_file():
    file = request.files['file']
    if file:
        filename = secure_filename(file.filename)
        file.save(filename)

        # dealing with mehods

        methods_new_data = process_file(filename)
        # Load the saved model
        #long_loaded_model = joblib.load('Random_Forest_embedding_method_level_model.pkl')
        # Prepare input tensors for new data
        #X_new = process_data_in_chunks(methods_new_data, chunk_size)

        methods_new_data['embeddings'] = methods_new_data['method_code_snippet'].apply(get_code_embeddings_batch)



                # Define the labels and corresponding models
        model_info= {
            'SVM': {
                'path': 'SVM_embedding_method_level_model.pkl',
                'scaler': 'SVM_embedding_method_level_scaler.pkl',
                'labels': ['comment_ratio', 'message_chain', 'feature_envy']
            },
            'Logistic_Regression': {
                'path': 'Logistic_Regression_embedding_method_level_model.pkl',
                'scaler': 'Logistic_Regression_embedding_method_level_scaler.pkl',
                'labels': ['large_parameter_list', 'small_method','middle_man','excessive_return_statement','complicated_boolean_expression']
            },
            'Random_Forest': {
                'path': 'Random_Forest_embedding_method_level_model.pkl',
                'scaler': 'Random_Forest_embedding_method_level_scaler.pkl',
                'labels': ['long_method']
            }

        }

        # Define the order of all labels
        all_labels = [
            'large_parameter_list', 'small_method',
            'message_chain', 'middle_man',
            'excessive_return_statement', 'comment_ratio',
            'complicated_boolean_expression', 'long_method',
            'feature_envy'
        ]

                # Function to get specific predictions
        def get_specific_predictions(y_pred, labels, all_labels):
            indices = [all_labels.index(label) for label in labels]
            return y_pred[:, indices]

        # Initialize an empty dictionary to store predictions
        predictions = {label: [] for label in all_labels}

        # Process each model and its corresponding labels
        for model_name, info in model_info.items():
            # Load the model and scaler
            model = joblib.load(info['path'])
            scaler = joblib.load(info['scaler'])

            X_new = np.stack(methods_new_data['embeddings'].values)
            X_new = X_new.reshape(X_new.shape[0], -1)

            # Transform the new data using the scaler
            X_new_data = scaler.transform(X_new)

            # Make predictions
            y_pred = model.predict(X_new_data)

            # Get specific predictions for the labels of the current model
            specific_y_pred = get_specific_predictions(y_pred, info['labels'], all_labels)

            # Store the predictions in the dictionary
            for i, label in enumerate(info['labels']):
                methods_new_data[f'is_{label}'] = specific_y_pred[:, i]
                methods_new_data[f'is_{label}'] = methods_new_data[f'is_{label}'].map({1: label.replace('_', ' ').title(), 0: f'Not {label.replace("_", " ").title()}'})

        #dealing with classes
        classes_new_data = god_process_file(filename)
        classes_new_data['count_methods'] = classes_new_data['class_code_snippet'].apply(count_methods)
        classes_new_data['count_fields'] = classes_new_data['class_code_snippet'].apply(count_fields)
        classes_new_data['tcc'] = classes_new_data['class_code_snippet'].apply(calculate_tcc)
        classes_new_data['loc'] = classes_new_data['class_code_snippet'].apply(count_lines_of_code)
        classes_new_data['atfd'] = classes_new_data['class_code_snippet'].apply(calculate_atfd)
        classes_new_data['cyclomatic_complexity'] = classes_new_data['class_code_snippet'].apply(calculate_cyclomatic_complexity)
        classes_new_data['lcom5'] = classes_new_data['class_code_snippet'].apply(calculate_lcom5)


        # Define the labels and corresponding models
        model_info = {
            'SVM': {
                'path': 'SVM_metrics_class_level_model.pkl',
                'scaler': 'SVM_metrics_class_level_scaler.pkl',
                'labels': ['data_class', 'god_class', 'imperative_loops']
            },
            'Logistic_Regression': {
                'path': 'Logistic_Regression_metrics_class_level_model.pkl',
                'scaler': 'Logistic_Regression_metrics_class_level_scaler.pkl',
                'labels': ['primitive_obsession', 'switch_statement']
            },
            'Random_Forest': {
                'path': 'Random_Forest_metrics_class_level_model.pkl',
                'scaler': 'Random_Forest_metrics_class_level_scaler.pkl',
                'labels': ['complicated_boolean_expression', 'divergent_change', 'excessive_return_statement', 'middle_man']
            },
            'Decision_Tree': {
                'path': 'Decision_Tree_metrics_class_level_model.pkl',
                'scaler': 'Decision_Tree_metrics_class_level_scaler.pkl',
                'labels': ['comment_ratio', 'inappropriate_intimacy', 'lazy_class', 'swiss_army_Knife']
            }
        }

        # Define the order of all labels
        all_labels = [
            'swiss_army_Knife', 'lazy_class', 'inappropriate_intimacy', 'switch_statement',
            'middle_man', 'primitive_obsession', 'imperative_loops', 'divergent_change',
            'excessive_return_statement', 'comment_ratio', 'complicated_boolean_expression',
            'god_class', 'data_class'
        ]

        # Function to get specific predictions
        def get_specific_predictions(y_pred, labels, all_labels):
            indices = [all_labels.index(label) for label in labels]
            return y_pred[:, indices]

        # Initialize an empty dictionary to store predictions
        predictions = {label: [] for label in all_labels}

        # Process each model and its corresponding labels
        for model_name, info in model_info.items():
            # Load the model and scaler
            model = joblib.load(info['path'])
            scaler = joblib.load(info['scaler'])

            # Define the features used for prediction
            features = ['count_methods', 'count_fields', 'tcc', 'cyclomatic_complexity', 'loc', 'atfd', 'lcom5']

            # Transform the new data using the scaler
            X_new_data = scaler.transform(classes_new_data[features])

            # Make predictions
            y_pred = model.predict(X_new_data)

            # Get specific predictions for the labels of the current model
            specific_y_pred = get_specific_predictions(y_pred, info['labels'], all_labels)

            # Store the predictions in the dictionary
            for i, label in enumerate(info['labels']):
                classes_new_data[f'is_{label}'] = specific_y_pred[:, i]
                classes_new_data[f'is_{label}'] = classes_new_data[f'is_{label}'].map({1: label.replace('_', ' ').title(), 0: f'Not {label.replace("_", " ").title()}'})

        # Render the template with the updated data
        return render_template('index.html', methods_result=methods_new_data.to_dict(orient='records'), classes_result=classes_new_data.to_dict(orient='records'))


    else:
        error = "No file selected."
        return render_template('index.html', error=error)

if __name__ == '__main__':
    # Set up the ngrok tunnel
    url = ngrok.connect(5000)
    print(f" * Tunnel URL: {url}")
    app.run(port=5000)

 * Tunnel URL: NgrokTunnel: "https://fd13-34-118-243-167.ngrok-free.app" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [18/Jul/2024 18:13:13] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [18/Jul/2024 18:13:14] "GET /favicon.ico HTTP/1.1" 404 -
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:432: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:432: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:432: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:432: UserWarning: X has feature names, but StandardScaler was fitted without feature name

In [ ]:
!pip install Flask pyngrok
!ngrok authtoken 2jG6PIOLKOFsTOVbCGR9hPCmUCX_4V9xU37QAKjiXTkkUqsqH
from flask import Flask
from pyngrok import ngrok

# Create a Flask app
app = Flask(__name__)

@app.route('/')
def hello_world():
    return 'Hello, World!'

# Run the Flask app
if __name__ == '__main__':
    # Set up the ngrok tunnel
    url = ngrok.connect(5000)
    print(f" * Tunnel URL: {url}")
    app.run(port=5000)


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
 * Tunnel URL: NgrokTunnel: "https://870c-34-86-156-72.ngrok-free.app" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [15/Jul/2024 13:46:15] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/Jul/2024 13:46:16] "GET /favicon.ico HTTP/1.1" 404 -


In [ ]:
#try class level only

In [4]:
!pip install flask-ngrok
!pip install pyngrok
!pip install torch transformers
!pip install transformers
!pip install javalang
import nltk
import pandas as pd
import torch
import javalang
import numpy as np
import joblib
import ast
from sklearn.preprocessing import StandardScaler
from javalang.ast import Node
from javalang.parser import Parser
from javalang.tokenizer import tokenize
from flask import Flask, render_template, request
from flask_ngrok import run_with_ngrok
from werkzeug.utils import secure_filename
from pyngrok import ngrok

In [7]:
# !pip install scikit-learn==0.24.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 78.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (pyproject.toml) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [ ]:

# for presenting if needed
# use all classifers


app = Flask(__name__, template_folder='/content/templates')
#ngrok.set_auth_token('2jG6PIOLKOFsTOVbCGR9hPCmUCX_4V9xU37QAKjiXTkkUqsqH')
# run_with_ngrok(app)

#preprocess class snippets
#count_methods
def count_methods(code):
    # Tokenize and parse the evaluated code
    tokens = javalang.tokenizer.tokenize(code)
    parser = javalang.parser.Parser(tokens)
    tree = parser.parse_member_declaration()
    method_count = 0
    for path, node in tree:
        if isinstance(node, javalang.tree.MethodDeclaration):
            method_count += 1
    return method_count

#count_fields
def count_fields(code):
    # Tokenize and parse the evaluated code
    tokens = javalang.tokenizer.tokenize(code)
    parser = javalang.parser.Parser(tokens)
    tree = parser.parse_member_declaration()
    field_count = 0
    for path, node in tree:
        if isinstance(node, javalang.tree.FieldDeclaration):
            field_count += len(node.declarators)
    return field_count

def calculate_tcc(code):
    # Tokenize and parse the evaluated code
    tokens = javalang.tokenizer.tokenize(code)
    parser = javalang.parser.Parser(tokens)
    tree = parser.parse_member_declaration()

    method_pairs = 0
    common_method_pairs = 0

    for path, node in tree:
        if isinstance(node, javalang.tree.MethodDeclaration):
            method_pairs += 1
            method_invocations = set()
            for _, invocation_node in node.filter(javalang.tree.MethodInvocation):
                method_invocations.add(invocation_node.member)

            for _, other_node in tree:
                if isinstance(other_node, javalang.tree.MethodDeclaration):
                    if other_node.name != node.name:
                        other_method_invocations = set()
                        for _, invocation_node in other_node.filter(javalang.tree.MethodInvocation):
                            other_method_invocations.add(invocation_node.member)
                        if method_invocations.intersection(other_method_invocations):
                            common_method_pairs += 1

    if method_pairs > 0:
        tcc = common_method_pairs / method_pairs
    else:
        tcc = 0

    return tcc


def count_lines_of_code(code):
    # Split the code into lines
    lines = code.split('\n')

    # Count the non-empty lines of code
    loc = sum(1 for line in lines if line.strip() != '')

    return loc

def calculate_atfd(code):
    # Tokenize and parse the evaluated code
    tokens = javalang.tokenizer.tokenize(code)
    parser = javalang.parser.Parser(tokens)
    tree = parser.parse_member_declaration()

    atfd = 0  # Initialize the ATFD metric to 0

    field_declarations = set()  # Store the field declarations in the class

    for path, node in tree:
        if isinstance(node, javalang.tree.FieldDeclaration):
            # Add the field names to the set of field declarations
            for declarator in node.declarators:
                field_declarations.add(declarator.name)

        if isinstance(node, javalang.tree.MethodDeclaration):
            for parameter in node.parameters:
                # Check if the parameter type is from an unrelated class
                if parameter.type.name not in field_declarations:
                    atfd += 1  # Increment the ATFD metric

            if node.body is not None:
                for statement in node.body:
                    # Check if the statement is a method invocation
                    if isinstance(statement, javalang.tree.MethodInvocation):
                        # Check if the invoked method belongs to an unrelated class
                        if statement.member not in field_declarations:
                            atfd += 1  # Increment the ATFD metric

    return atfd



def calculate_cyclomatic_complexity(code):
    try:
        # Tokenize and parse the evaluated code
        tokens = javalang.tokenizer.tokenize(code)
        parser = javalang.parser.Parser(tokens)
        tree = parser.parse_member_declaration()

        # Count the number of decision points (branches and loops)
        complexity = 1  # Start with a complexity of 1 for the method itself
        for _, node in tree:
            if isinstance(node, javalang.tree.IfStatement):
                complexity += 1
            elif isinstance(node, javalang.tree.ForStatement):
                complexity += 1
            elif isinstance(node, javalang.tree.WhileStatement):
                complexity += 1
            elif isinstance(node, javalang.tree.DoStatement):
                complexity += 1
            elif isinstance(node, javalang.tree.SwitchStatement):
                complexity += len(node.cases)

        return complexity

    except (SyntaxError, ValueError) as e:
        print("Error evaluating code:")
        print(code)
        raise e


def calculate_lcom5(code):
    # Tokenize and parse the code snippet
    tokens = javalang.tokenizer.tokenize(code)
    parser = javalang.parser.Parser(tokens)
    tree = parser.parse_member_declaration()

    # Collect all the method names and instance variables
    methods = {}
    variables = set()
    for _, node in tree:
        if isinstance(node, javalang.tree.MethodDeclaration):
            methods[node.name] = set()
            for path, child_node in node:
                if isinstance(child_node, javalang.tree.VariableDeclarator):
                    variables.add(child_node.name)
                    methods[node.name].add(child_node.name)

    # Calculate the LCOM5 metric
    lcom5 = 0
    for method1, vars1 in methods.items():
        for method2, vars2 in methods.items():
            if method1 != method2:
                common_vars = vars1.intersection(vars2)
                if len(common_vars) == 0:
                    lcom5 += 1

    return lcom5





# Preprocess methods snippets
# Load the model and tokenizer
model_name = "claudios/cubert-20210711-Java-1024"
embedding_model = AutoModel.from_pretrained(model_name)
java_tokenizer = AutoTokenizer.from_pretrained("CAUKiel/JavaBERT")

# Ensure the model is in evaluation mode
embedding_model.eval()

# Check if GPU is available and move the model to GPU if possible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
embedding_model.to(device)

# Function to get embeddings for a batch of code snippets
def get_code_embeddings_batch(code_snippets):
    inputs = java_tokenizer(code_snippets, return_tensors="pt", truncation=True, padding='max_length', max_length=1024)
    inputs = {key: value.to(device) for key, value in inputs.items()}
    with torch.no_grad():
        outputs = embedding_model(**inputs)
    embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()  # Use mean pooling for a fixed-size representation
    return embeddings



def get_method_start_end(method_node, tree):
    startpos = None
    endpos = None
    startline = None
    endline = None
    for path, node in tree:
        if startpos is not None and method_node not in path:
            endpos = node.position
            endline = node.position.line if node.position is not None else None
            break
        if startpos is None and node == method_node:
            startpos = node.position
            startline = node.position.line if node.position is not None else None
    return startpos, endpos, startline, endline

def get_method_text(startpos, endpos, startline, endline, last_endline_index, codelines):
    if startpos is None:
        return "", None, None, None
    else:
        startline_index = startline - 1
        endline_index = endline - 1 if endpos is not None else None

        # 1. check for and fetch annotations
        if last_endline_index is not None:
            for line in codelines[(last_endline_index + 1):(startline_index)]:
                if "@" in line:
                    startline_index = startline_index - 1
        meth_text = "<ST>".join(codelines[startline_index:endline_index])
        meth_text = meth_text[:meth_text.rfind("}") + 1]

        # 2. remove trailing rbrace for last methods & any external content/comments
        # if endpos is None and
        if not abs(meth_text.count("}") - meth_text.count("{")) == 0:
            # imbalanced braces
            brace_diff = abs(meth_text.count("}") - meth_text.count("{"))

            for _ in range(brace_diff):
                meth_text = meth_text[:meth_text.rfind("}")]
                meth_text = meth_text[:meth_text.rfind("}") + 1]

        meth_lines = meth_text.split("<ST>")
        meth_text = "".join(meth_lines)
        last_endline_index = startline_index + (len(meth_lines) - 1)

        return meth_text, (startline_index + 1), (last_endline_index + 1), last_endline_index

def process_file(file_path):
    with open(file_path, 'r') as r:
        codelines = r.readlines()
        code_text = ''.join(codelines)

    tree = javalang.parse.parse(code_text)
    methods = {}
    lex = None
    for _, method_node in tree.filter(javalang.tree.MethodDeclaration):
        startpos, endpos, startline, endline = get_method_start_end(method_node, tree)
        method_text, startline, endline, lex = get_method_text(startpos, endpos, startline, endline, lex, codelines)
        methods[method_node.name] = method_text

    # Create a new DataFrame to store the extracted methods
    methods_new_data = pd.DataFrame(columns=['method_name', 'method_code_snippet'])
    methods_new_data['method_name'] = list(methods.keys())
    methods_new_data['method_code_snippet'] = list(methods.values())

    return methods_new_data

def get_class_body(class_node, codelines):
    class_start = class_node.position.line - 1
    brace_count = 0

    for index in range(class_start, len(codelines)):
        line = codelines[index]
        brace_count += line.count('{')
        brace_count -= line.count('}')
        if brace_count == 0:
            class_end = index + 1
            break

    class_body = '\n'.join(codelines[class_start:class_end])
    return class_body.strip()

def god_process_file(file_path):
    with open(file_path, 'r') as r:
      codelines = r.readlines()
      code_text = ''.join(codelines)

    tree = javalang.parse.parse(code_text)
    classes = []

    for _, class_node in tree.filter(javalang.tree.ClassDeclaration):
      class_name = class_node.name
      class_body = get_class_body(class_node, codelines)
      classes.append({'class_name': class_name, 'class_code_snippet': class_body})

    classes_new_data = pd.DataFrame(classes)
    return classes_new_data





#preprocess methods snippets
# Load the model and tokenizer


# Define the order of all class-level labels
class_labels = [
    'swiss_army_Knife', 'lazy_class', 'inappropriate_intimacy', 'switch_statement',
    'middle_man', 'primitive_obsession', 'imperative_loops', 'divergent_change',
    'excessive_return_statement', 'comment_ratio', 'complicated_boolean_expression',
    'god_class', 'data_class'
]

# Define the order of all method-level labels
method_labels = [
    'large_parameter_list', 'small_method', 'message_chain', 'middle_man',
    'excessive_return_statement', 'comment_ratio', 'complicated_boolean_expression',
    'long_method', 'feature_envy'
]

# Define the labels and corresponding models for class level
class_model_info = {
    'SVM': {
        'path': 'SVM_metrics_class_level_model.pkl',
        'scaler': 'SVM_metrics_class_level_scaler.pkl',
        'labels': ['data_class', 'god_class', 'imperative_loops']
    },
    'Logistic_Regression': {
        'path': 'Logistic_Regression_metrics_class_level_model.pkl',
        'scaler': 'Logistic_Regression_metrics_class_level_scaler.pkl',
        'labels': ['primitive_obsession', 'switch_statement']
    },
    'Random_Forest': {
        'path': 'Random_Forest_metrics_class_level_model.pkl',
        'scaler': 'Random_Forest_metrics_class_level_scaler.pkl',
        'labels': ['complicated_boolean_expression', 'divergent_change', 'excessive_return_statement', 'middle_man']
    },
    'Decision_Tree': {
        'path': 'Decision_Tree_metrics_class_level_model.pkl',
        'scaler': 'Decision_Tree_metrics_class_level_scaler.pkl',
        'labels': ['comment_ratio', 'inappropriate_intimacy', 'lazy_class', 'swiss_army_Knife']
    }
}

# Define the labels and corresponding models for method level
method_model_info = {
    'SVM': {
        'path': 'SVM_embedding_method_level_model.pkl',
        'scaler': 'SVM_embedding_method_level_scaler.pkl',
        'labels': ['comment_ratio', 'message_chain', 'feature_envy']
    },
    'Logistic_Regression': {
        'path': 'Logistic_Regression_embedding_method_level_model.pkl',
        'scaler': 'Logistic_Regression_embedding_method_level_scaler.pkl',
        'labels': ['large_parameter_list', 'small_method', 'middle_man', 'excessive_return_statement', 'complicated_boolean_expression']
    },
    'Random_Forest': {
        'path': 'Random_Forest_embedding_method_level_model.pkl',
        'scaler': 'Random_Forest_embedding_method_level_scaler.pkl',
        'labels': ['long_method']
    }
}

# Function to get specific predictions
def get_specific_predictions(y_pred, labels, all_labels):
    indices = [all_labels.index(label) for label in labels]
    return y_pred[:, indices]

@app.route('/')
def index():
    return render_template('index.html')

@app.route('/upload', methods=['POST'])
def upload_file():
    file = request.files['file']
    if file:
        filename = secure_filename(file.filename)
        file.save(filename)

        # dealing with methods
        methods_new_data = process_file(filename)

        # Get embeddings for method snippets
        code_snippets = methods_new_data['method_code_snippet'].tolist()
        method_embeddings = get_code_embeddings_batch(code_snippets)

        # Initialize an empty dictionary to store method-level predictions
        method_predictions = {label: [] for label in method_labels}

        # Process each method model and its corresponding labels
        for model_name, info in method_model_info.items():
            # Load the model and scaler
            model = joblib.load(info['path'])
            scaler = joblib.load(info['scaler'])

            # Transform the embeddings using the scaler
            X_new_data = scaler.transform(method_embeddings)

            # Make predictions
            y_pred = model.predict(X_new_data)

            # Get specific predictions for the labels of the current model
            specific_y_pred = get_specific_predictions(y_pred, info['labels'], method_labels)

            # Store the predictions in the dictionary
            for i, label in enumerate(info['labels']):
                methods_new_data[f'is_{label}'] = specific_y_pred[:, i]
                methods_new_data[f'is_{label}'] = methods_new_data[f'is_{label}'].map({1: label.replace('_', ' ').title(), 0: f'Not {label.replace("_", " ").title()}'})

        # dealing with classes
        classes_new_data = god_process_file(filename)
        classes_new_data['count_methods'] = classes_new_data['class_code_snippet'].apply(count_methods)
        classes_new_data['count_fields'] = classes_new_data['class_code_snippet'].apply(count_fields)
        classes_new_data['tcc'] = classes_new_data['class_code_snippet'].apply(calculate_tcc)
        classes_new_data['loc'] = classes_new_data['class_code_snippet'].apply(count_lines_of_code)
        classes_new_data['atfd'] = classes_new_data['class_code_snippet'].apply(calculate_atfd)
        classes_new_data['cyclomatic_complexity'] = classes_new_data['class_code_snippet'].apply(calculate_cyclomatic_complexity)
        classes_new_data['lcom5'] = classes_new_data['class_code_snippet'].apply(calculate_lcom5)

        # Initialize an empty dictionary to store class-level predictions
        class_predictions = {label: [] for label in class_labels}

        # Process each class model and its corresponding labels
        for model_name, info in class_model_info.items():
            # Load the model and scaler
            model = joblib.load(info['path'])
            scaler = joblib.load(info['scaler'])

            # Define the features used for prediction
            features = ['count_methods', 'count_fields', 'tcc', 'cyclomatic_complexity', 'loc', 'atfd', 'lcom5']

            # Transform the new data using the scaler
            X_new_data = scaler.transform(classes_new_data[features])

            # Make predictions
            y_pred = model.predict(X_new_data)

            # Get specific predictions for the labels of the current model
            specific_y_pred = get_specific_predictions(y_pred, info['labels'], class_labels)

            # Store the predictions in the dictionary
            for i, label in enumerate(info['labels']):
                classes_new_data[f'is_{label}'] = specific_y_pred[:, i]
                classes_new_data[f'is_{label}'] = classes_new_data[f'is_{label}'].map({1: label.replace('_', ' ').title(), 0: f'Not {label.replace("_", " ").title()}'})

        # Render the template with the updated data
        return render_template('index.html', methods_result=methods_new_data.to_dict(orient='records'), classes_result=classes_new_data.to_dict(orient='records'))

    else:
        error = "No file selected."
        return render_template('index.html', error=error)

if __name__ == '__main__':
    # Set up the ngrok tunnel
    url = ngrok.connect(5000)
    print(f" * Tunnel URL: {url}")
    app.run(port=5000)

In [10]:
#run for class models and method models
#but SVM for method take some time to upload
# also random forrest and decisiontree pkl have incompatable issue with sicket learn
#so (you can use logistic as temporiry solution)

app = Flask(__name__, template_folder='/content/templates')
#ngrok.set_auth_token('2jG6PIOLKOFsTOVbCGR9hPCmUCX_4V9xU37QAKjiXTkkUqsqH')
# run_with_ngrok(app)

#preprocess class snippets
#count_methods
def count_methods(code):
    # Tokenize and parse the evaluated code
    tokens = javalang.tokenizer.tokenize(code)
    parser = javalang.parser.Parser(tokens)
    tree = parser.parse_member_declaration()
    method_count = 0
    for path, node in tree:
        if isinstance(node, javalang.tree.MethodDeclaration):
            method_count += 1
    return method_count

#count_fields
def count_fields(code):
    # Tokenize and parse the evaluated code
    tokens = javalang.tokenizer.tokenize(code)
    parser = javalang.parser.Parser(tokens)
    tree = parser.parse_member_declaration()
    field_count = 0
    for path, node in tree:
        if isinstance(node, javalang.tree.FieldDeclaration):
            field_count += len(node.declarators)
    return field_count

def calculate_tcc(code):
    # Tokenize and parse the evaluated code
    tokens = javalang.tokenizer.tokenize(code)
    parser = javalang.parser.Parser(tokens)
    tree = parser.parse_member_declaration()

    method_pairs = 0
    common_method_pairs = 0

    for path, node in tree:
        if isinstance(node, javalang.tree.MethodDeclaration):
            method_pairs += 1
            method_invocations = set()
            for _, invocation_node in node.filter(javalang.tree.MethodInvocation):
                method_invocations.add(invocation_node.member)

            for _, other_node in tree:
                if isinstance(other_node, javalang.tree.MethodDeclaration):
                    if other_node.name != node.name:
                        other_method_invocations = set()
                        for _, invocation_node in other_node.filter(javalang.tree.MethodInvocation):
                            other_method_invocations.add(invocation_node.member)
                        if method_invocations.intersection(other_method_invocations):
                            common_method_pairs += 1

    if method_pairs > 0:
        tcc = common_method_pairs / method_pairs
    else:
        tcc = 0

    return tcc


def count_lines_of_code(code):
    # Split the code into lines
    lines = code.split('\n')

    # Count the non-empty lines of code
    loc = sum(1 for line in lines if line.strip() != '')

    return loc

def calculate_atfd(code):
    # Tokenize and parse the evaluated code
    tokens = javalang.tokenizer.tokenize(code)
    parser = javalang.parser.Parser(tokens)
    tree = parser.parse_member_declaration()

    atfd = 0  # Initialize the ATFD metric to 0

    field_declarations = set()  # Store the field declarations in the class

    for path, node in tree:
        if isinstance(node, javalang.tree.FieldDeclaration):
            # Add the field names to the set of field declarations
            for declarator in node.declarators:
                field_declarations.add(declarator.name)

        if isinstance(node, javalang.tree.MethodDeclaration):
            for parameter in node.parameters:
                # Check if the parameter type is from an unrelated class
                if parameter.type.name not in field_declarations:
                    atfd += 1  # Increment the ATFD metric

            if node.body is not None:
                for statement in node.body:
                    # Check if the statement is a method invocation
                    if isinstance(statement, javalang.tree.MethodInvocation):
                        # Check if the invoked method belongs to an unrelated class
                        if statement.member not in field_declarations:
                            atfd += 1  # Increment the ATFD metric

    return atfd



def calculate_cyclomatic_complexity(code):
    try:
        # Tokenize and parse the evaluated code
        tokens = javalang.tokenizer.tokenize(code)
        parser = javalang.parser.Parser(tokens)
        tree = parser.parse_member_declaration()

        # Count the number of decision points (branches and loops)
        complexity = 1  # Start with a complexity of 1 for the method itself
        for _, node in tree:
            if isinstance(node, javalang.tree.IfStatement):
                complexity += 1
            elif isinstance(node, javalang.tree.ForStatement):
                complexity += 1
            elif isinstance(node, javalang.tree.WhileStatement):
                complexity += 1
            elif isinstance(node, javalang.tree.DoStatement):
                complexity += 1
            elif isinstance(node, javalang.tree.SwitchStatement):
                complexity += len(node.cases)

        return complexity

    except (SyntaxError, ValueError) as e:
        print("Error evaluating code:")
        print(code)
        raise e


def calculate_lcom5(code):
    # Tokenize and parse the code snippet
    tokens = javalang.tokenizer.tokenize(code)
    parser = javalang.parser.Parser(tokens)
    tree = parser.parse_member_declaration()

    # Collect all the method names and instance variables
    methods = {}
    variables = set()
    for _, node in tree:
        if isinstance(node, javalang.tree.MethodDeclaration):
            methods[node.name] = set()
            for path, child_node in node:
                if isinstance(child_node, javalang.tree.VariableDeclarator):
                    variables.add(child_node.name)
                    methods[node.name].add(child_node.name)

    # Calculate the LCOM5 metric
    lcom5 = 0
    for method1, vars1 in methods.items():
        for method2, vars2 in methods.items():
            if method1 != method2:
                common_vars = vars1.intersection(vars2)
                if len(common_vars) == 0:
                    lcom5 += 1

    return lcom5





# Preprocess methods snippets
# Load the model and tokenizer
model_name = "claudios/cubert-20210711-Java-1024"
embedding_model = AutoModel.from_pretrained(model_name)
java_tokenizer = AutoTokenizer.from_pretrained("CAUKiel/JavaBERT")

# Ensure the model is in evaluation mode
embedding_model.eval()

# Check if GPU is available and move the model to GPU if possible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
embedding_model.to(device)

# Function to get embeddings for a batch of code snippets
def get_code_embeddings_batch(code_snippets):
    inputs = java_tokenizer(code_snippets, return_tensors="pt", truncation=True, padding='max_length', max_length=1024)
    inputs = {key: value.to(device) for key, value in inputs.items()}
    with torch.no_grad():
        outputs = embedding_model(**inputs)
    embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()  # Use mean pooling for a fixed-size representation
    return embeddings



def get_method_start_end(method_node, tree):
    startpos = None
    endpos = None
    startline = None
    endline = None
    for path, node in tree:
        if startpos is not None and method_node not in path:
            endpos = node.position
            endline = node.position.line if node.position is not None else None
            break
        if startpos is None and node == method_node:
            startpos = node.position
            startline = node.position.line if node.position is not None else None
    return startpos, endpos, startline, endline

def get_method_text(startpos, endpos, startline, endline, last_endline_index, codelines):
    if startpos is None:
        return "", None, None, None
    else:
        startline_index = startline - 1
        endline_index = endline - 1 if endpos is not None else None

        # 1. check for and fetch annotations
        if last_endline_index is not None:
            for line in codelines[(last_endline_index + 1):(startline_index)]:
                if "@" in line:
                    startline_index = startline_index - 1
        meth_text = "<ST>".join(codelines[startline_index:endline_index])
        meth_text = meth_text[:meth_text.rfind("}") + 1]

        # 2. remove trailing rbrace for last methods & any external content/comments
        # if endpos is None and
        if not abs(meth_text.count("}") - meth_text.count("{")) == 0:
            # imbalanced braces
            brace_diff = abs(meth_text.count("}") - meth_text.count("{"))

            for _ in range(brace_diff):
                meth_text = meth_text[:meth_text.rfind("}")]
                meth_text = meth_text[:meth_text.rfind("}") + 1]

        meth_lines = meth_text.split("<ST>")
        meth_text = "".join(meth_lines)
        last_endline_index = startline_index + (len(meth_lines) - 1)

        return meth_text, (startline_index + 1), (last_endline_index + 1), last_endline_index

def process_file(file_path):
    with open(file_path, 'r') as r:
        codelines = r.readlines()
        code_text = ''.join(codelines)

    tree = javalang.parse.parse(code_text)
    methods = {}
    lex = None
    for _, method_node in tree.filter(javalang.tree.MethodDeclaration):
        startpos, endpos, startline, endline = get_method_start_end(method_node, tree)
        method_text, startline, endline, lex = get_method_text(startpos, endpos, startline, endline, lex, codelines)
        methods[method_node.name] = method_text

    # Create a new DataFrame to store the extracted methods
    methods_new_data = pd.DataFrame(columns=['method_name', 'method_code_snippet'])
    methods_new_data['method_name'] = list(methods.keys())
    methods_new_data['method_code_snippet'] = list(methods.values())

    return methods_new_data

def get_class_body(class_node, codelines):
    class_start = class_node.position.line - 1
    brace_count = 0

    for index in range(class_start, len(codelines)):
        line = codelines[index]
        brace_count += line.count('{')
        brace_count -= line.count('}')
        if brace_count == 0:
            class_end = index + 1
            break

    class_body = '\n'.join(codelines[class_start:class_end])
    return class_body.strip()

def god_process_file(file_path):
    with open(file_path, 'r') as r:
      codelines = r.readlines()
      code_text = ''.join(codelines)

    tree = javalang.parse.parse(code_text)
    classes = []

    for _, class_node in tree.filter(javalang.tree.ClassDeclaration):
      class_name = class_node.name
      class_body = get_class_body(class_node, codelines)
      classes.append({'class_name': class_name, 'class_code_snippet': class_body})

    classes_new_data = pd.DataFrame(classes)
    return classes_new_data





#preprocess methods snippets
# Load the model and tokenizer


# Define the order of all class-level labels
class_labels = [
    'swiss_army_Knife', 'lazy_class', 'inappropriate_intimacy', 'switch_statement',
    'middle_man', 'primitive_obsession', 'imperative_loops', 'divergent_change',
    'excessive_return_statement', 'comment_ratio', 'complicated_boolean_expression',
    'god_class', 'data_class'
]

# Define the order of all method-level labels
method_labels = [
    'large_parameter_list', 'small_method', 'message_chain', 'middle_man',
    'excessive_return_statement', 'comment_ratio', 'complicated_boolean_expression',
    'long_method', 'feature_envy'
]

# Define the labels and corresponding models for class level
class_model_info = {
    'SVM': {
        'path': 'Logistic_Regression_metrics_class_level_model.pkl',
        'scaler': 'Logistic_Regression_metrics_class_level_scaler.pkl',
        'labels': ['data_class', 'god_class', 'imperative_loops']
    },
    'Logistic_Regression': {
        'path': 'Logistic_Regression_metrics_class_level_model.pkl',
        'scaler': 'Logistic_Regression_metrics_class_level_scaler.pkl',
        'labels': ['primitive_obsession', 'switch_statement']
    },
    'Random_Forest': {
        'path': 'Logistic_Regression_metrics_class_level_model.pkl',
        'scaler': 'Logistic_Regression_metrics_class_level_scaler.pkl',
        'labels': ['complicated_boolean_expression', 'divergent_change', 'excessive_return_statement', 'middle_man']
    },
    'Decision_Tree': {
        'path': 'Logistic_Regression_metrics_class_level_model.pkl',
        'scaler': 'Logistic_Regression_metrics_class_level_scaler.pkl',
        'labels': ['comment_ratio', 'inappropriate_intimacy', 'lazy_class', 'swiss_army_Knife']
    }
}

# Define the labels and corresponding models for method level
method_model_info = {
    'SVM': {
        'path': 'Logistic_Regression_embedding_method_level_model.pkl',
        'scaler': 'Logistic_Regression_embedding_method_level_scaler.pkl',
        'labels': ['comment_ratio', 'message_chain', 'feature_envy']
    },
    'Logistic_Regression': {
        'path': 'Logistic_Regression_embedding_method_level_model.pkl',
        'scaler': 'Logistic_Regression_embedding_method_level_scaler.pkl',
        'labels': ['large_parameter_list', 'small_method', 'middle_man', 'excessive_return_statement', 'complicated_boolean_expression']
    },
    'Random_Forest': {
        'path': 'Logistic_Regression_embedding_method_level_model.pkl',
        'scaler': 'Logistic_Regression_embedding_method_level_scaler.pkl',
        'labels': ['long_method']
    }
}

# Function to get specific predictions
def get_specific_predictions(y_pred, labels, all_labels):
    indices = [all_labels.index(label) for label in labels]
    return y_pred[:, indices]

@app.route('/')
def index():
    return render_template('index.html')

@app.route('/upload', methods=['POST'])
def upload_file():
    file = request.files['file']
    if file:
        filename = secure_filename(file.filename)
        file.save(filename)

        # dealing with methods
        methods_new_data = process_file(filename)

        # Get embeddings for method snippets
        code_snippets = methods_new_data['method_code_snippet'].tolist()
        method_embeddings = get_code_embeddings_batch(code_snippets)

        # Initialize an empty dictionary to store method-level predictions
        method_predictions = {label: [] for label in method_labels}

        # Process each method model and its corresponding labels
        for model_name, info in method_model_info.items():
            # Load the model and scaler
            model = joblib.load(info['path'])
            scaler = joblib.load(info['scaler'])

            # Transform the embeddings using the scaler
            X_new_data = scaler.transform(method_embeddings)

            # Make predictions
            y_pred = model.predict(X_new_data)

            # Get specific predictions for the labels of the current model
            specific_y_pred = get_specific_predictions(y_pred, info['labels'], method_labels)

            # Store the predictions in the dictionary
            for i, label in enumerate(info['labels']):
                methods_new_data[f'is_{label}'] = specific_y_pred[:, i]
                methods_new_data[f'is_{label}'] = methods_new_data[f'is_{label}'].map({1: label.replace('_', ' ').title(), 0: f'Not {label.replace("_", " ").title()}'})

        # dealing with classes
        classes_new_data = god_process_file(filename)
        classes_new_data['count_methods'] = classes_new_data['class_code_snippet'].apply(count_methods)
        classes_new_data['count_fields'] = classes_new_data['class_code_snippet'].apply(count_fields)
        classes_new_data['tcc'] = classes_new_data['class_code_snippet'].apply(calculate_tcc)
        classes_new_data['loc'] = classes_new_data['class_code_snippet'].apply(count_lines_of_code)
        classes_new_data['atfd'] = classes_new_data['class_code_snippet'].apply(calculate_atfd)
        classes_new_data['cyclomatic_complexity'] = classes_new_data['class_code_snippet'].apply(calculate_cyclomatic_complexity)
        classes_new_data['lcom5'] = classes_new_data['class_code_snippet'].apply(calculate_lcom5)

        # Initialize an empty dictionary to store class-level predictions
        class_predictions = {label: [] for label in class_labels}

        # Process each class model and its corresponding labels
        for model_name, info in class_model_info.items():
            # Load the model and scaler
            model = joblib.load(info['path'])
            scaler = joblib.load(info['scaler'])

            # Define the features used for prediction
            features = ['count_methods', 'count_fields', 'tcc', 'cyclomatic_complexity', 'loc', 'atfd', 'lcom5']

            # Transform the new data using the scaler
            X_new_data = scaler.transform(classes_new_data[features])

            # Make predictions
            y_pred = model.predict(X_new_data)

            # Get specific predictions for the labels of the current model
            specific_y_pred = get_specific_predictions(y_pred, info['labels'], class_labels)

            # Store the predictions in the dictionary
            for i, label in enumerate(info['labels']):
                classes_new_data[f'is_{label}'] = specific_y_pred[:, i]
                classes_new_data[f'is_{label}'] = classes_new_data[f'is_{label}'].map({1: label.replace('_', ' ').title(), 0: f'Not {label.replace("_", " ").title()}'})

        # Render the template with the updated data
        return render_template('index.html', methods_result=methods_new_data.to_dict(orient='records'), classes_result=classes_new_data.to_dict(orient='records'))

    else:
        error = "No file selected."
        return render_template('index.html', error=error)

if __name__ == '__main__':
    # Set up the ngrok tunnel
    url = ngrok.connect(5000)
    print(f" * Tunnel URL: {url}")
    app.run(port=5000)

 * Tunnel URL: NgrokTunnel: "https://1c35-34-32-183-70.ngrok-free.app" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [21/Jun/2025 08:28:53] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [21/Jun/2025 08:28:54] "GET /favicon.ico HTTP/1.1" 404 -
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.2.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MultiOutputClassifier from version 1.2.2 when using version 1.6.1. This might lead to breaking code or invalid results. Us